# Lab 09: How to answer follow-up questions with retrieval

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls) | In class**

## Goal

Answer a follow-up question by searching with a query that stands on its own, and check that query in code before you
trust it. The moment that teaches it: after a question about Boreal, the user asks *How long is the link valid?*
Searched as typed, it finds the Atlas policy, and the answer gives Atlas's time with a correct citation. Rewriting
the question to name Boreal fixes it; letting the model write its own search drops Boreal again, and a two-line check
catches both.

## What you will learn

- Why a follow-up searched as typed can find the wrong evidence, or leave the answer with no question, while every
  citation check passes.
- How to rewrite a follow-up into a standalone query with one chat call, and check the query in code: the product
  kept, no number the user did not type.
- How to let the model write the query itself as a search-tool call (`@tool` and `bind_tools`), and run the call.
- How to test both routes on a set of conversations over several runs, and see where an earlier answer leaks in.
- When to ask the user which product they mean instead of guessing, and why that rule belongs in code.

**In class:** Parts A to C. Parts D, E and F are self-study.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-24
OPENAI_API_KEY    found in .env


This lab uses `chat_model` and `embed_model` from the lines above, and `JSON_MODE` from `models.env`. It embeds the
corpus once (about 21,000 tokens) and makes about 120 short chat calls, well under one US cent on `gpt-4o-mini`. The
saved outputs and every count in the text come from `gpt-4o-mini`; the README says what changed with `gpt-6-luna`. Your own run will differ a little: model replies vary from run to run and similarity scores can move in the third decimal, so compare the order of results and the kind of reply with the text, not every number.

A **follow-up** is a question that leans on the turns before it: *What about Boreal?* means nothing alone. The
pipeline from Labs 02 and 05 searches with one question. This lab adds the step that turns a conversation's last turn
into a question the search can use, and checks that say whether it kept what the user meant.

---

## Part A: Search a follow-up as typed

Part A runs the Lab 05 pipeline on a conversation's last turn, as it would on any single question.

The conversations in this lab's `resources/conversations.json` are constructed: short help-desk exchanges about the
corpus's two products, Atlas and Boreal, with each earlier assistant turn written from one policy and citing it. Each
also records the product a search must keep and the policy that answers the last turn, an answer key the model never
sees. Load them:

In [3]:
import json

LAB = ROOT / "09_Follow_Up_Questions"
conversations = json.loads((LAB / "resources" / "conversations.json").read_text(encoding="utf-8"))["conversations"]
by_id = {case["id"]: case for case in conversations}
for case in conversations:
    print(f"{case['id']:18}{case['slice']:22}{case['products']!s:20}{case['source_id']!s:6}{case['turns'][-1]['content']}")

topic_switch      topic switch          ['Boreal']          H04   What about Boreal?
link_after_boreal reference resolution  ['Boreal']          H04   How long is the link valid?
account_type      reference resolution  ['Atlas']           H01   Can temporary accounts do that?
pronoun           reference resolution  ['Atlas']           H03   Who can export all of them?
earlier_answer    reference resolution  ['Atlas']           H05   Where can I download them?
evidence_gap      evidence gap          ['Boreal']          None  What about Boreal?
two_products      ambiguous             ['Atlas', 'Boreal'] None  How long is the link valid?


Seven conversations, each named by the kind of follow-up it tests (its **slice**). Five have a policy that answers the
last turn; `evidence_gap` has none (no policy covers Boreal's billing), and in `two_products` the answer depends on
which product the user means.

Build Lab 05's vector store with `shared/lc_pipeline.py`, and print the conversation this part follows:

In [4]:
from shared import lc_pipeline

store = lc_pipeline.build_store(lc_pipeline.to_documents(lc_pipeline.load_corpus()), settings)
case = by_id["link_after_boreal"]
for turn in case["turns"]:
    print(f"{turn['role']:>9}: {turn['content']}")

     user: How do I reset my Boreal password?
assistant: Boreal password reset uses a mobile app [H04].
     user: How long is the link valid?


The user asked about Boreal, got an answer, and now asks *How long is the link valid?*, meaning Boreal's reset
link. Search with that last turn as typed. `show()` prints each hit's score, ID and the start of its text:

In [5]:
def show(hits):
    for document, score in hits:
        print(f"  {score:.4f}  {document.id:13}{document.page_content[:70]}")

follow_up = case["turns"][-1]["content"]
hits = lc_pipeline.retrieve(store, follow_up)
show(hits)

  0.5949  H02          Atlas reset links expire after 30 minutes. Request a new link if the o
  0.3770  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.1892  H01          Atlas passwords can be reset by email. Temporary accounts require a su


The Atlas policy H02 comes first (about 0.59) and Boreal's H04 second (about 0.38). The search reads only the words it is
given, and *the link* fits Atlas's reset link better than Boreal's. Now answer it as Lab 05 does, ten times, with Lab
03's quote check on every claim. `answers()` groups identical answers and prints how often each came back:

In [6]:
from shared.answer import quote_found

from collections import Counter

def answers(question, hits, runs=3):
    lines = lc_pipeline.evidence(hits)
    seen = Counter()
    for _ in range(runs):
        reply = lc_pipeline.answer(question, lines, settings)
        seen[f"{reply.answer} | cited, quote found: {[(c.source_id, quote_found(c.model_dump(), lines)) for c in reply.claims]}"] += 1
    for text, count in seen.items():
        print(f"  {count} of {runs}: {text}")

answers(follow_up, hits, runs=10)

  1 of 10: Insufficient evidence. | cited, quote found: []
  7 of 10: Atlas reset links expire after 30 minutes. | cited, quote found: [('H02', True)]
  2 of 10: Atlas reset links expire after 30 minutes and Boreal reset links expire after 10 minutes. | cited, quote found: [('H02', True), ('H04', True)]


7 of 10 answers say *Atlas reset links expire after 30 minutes.*, citing H02 with its quote found; 2 give
Atlas's and Boreal's times together, and 1 refuses. None answers for Boreal alone. The cited answers pass every check
from Lab 02 and are about the wrong product, or both: nothing in the pipeline knew the user meant Boreal.

The first conversation fails another way. The user asked how long an Atlas reset link stays valid, then *What about
Boreal?*:

In [7]:
switch = by_id["topic_switch"]
switch_hits = lc_pipeline.retrieve(store, switch["turns"][-1]["content"])
show(switch_hits)
answers(switch["turns"][-1]["content"], switch_hits)

  0.3256  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.3202  S16-P01-C01  rainforest spanned a relatively narrow band. It expanded again during 
  0.2958  S16-P01-C00  Following the Cretaceous–Paleogene extinction event, the extinction of


  3 of 3: Insufficient evidence. | cited, quote found: []


This time H04 comes first, but only just: about 0.33 against 0.32 for a chunk about the Amazon rainforest. *Boreal* is
the only word in the turn the search can use, and it is also an English word for northern forests, which may be why
forest text ranks beside it. The answer refuses 3 of 3, with the right policy in front of it, because *What about
Boreal?* asks nothing.

So the last turn alone can lose the product (the search finds Atlas) or the task (the answer has no question). Both
come from one gap: the pipeline sees one turn, and the meaning is spread over three.

### Checkpoint A

| You should see | Our run |
|---|---|
| Conversations | 7, from `topic_switch` to `two_products` |
| *How long is the link valid?* as typed | H02 about 0.59, H04 0.38, H01 0.19 |
| Its answers | Atlas's 30 minutes with H02, quote found (7 of 10); both products' times (2 of 10); *Insufficient evidence.* (1 of 10) |
| *What about Boreal?* as typed | H04 about 0.33, just above S16-P01-C01 (0.32, *rainforest spanned ...*); *Insufficient evidence.* 3 of 3 |

---

## Part B: Rewrite the follow-up as a standalone query

Part B adds one chat call before the search. It turns the conversation into a **standalone query**: a question that
means what the last turn means and can be understood without the turns before it.

The model needs the conversation as text. Write each turn as `role: content`, one per line:

In [8]:
def history_text(turns):
    return "\n".join(f"{turn['role']}: {turn['content']}" for turn in turns)

print(history_text(case["turns"]))

user: How do I reset my Boreal password?
assistant: Boreal password reset uses a mobile app [H04].
user: How long is the link valid?


The instruction says what to keep and what not to do. A rewrite that answers the question (*Boreal links expire
after 10 minutes*) would search for its own guess, and a guess that matches a policy's words would look supported.
`lc_chat_model()` from Lab 05 builds the chat model from `models.env`. This is one plain call, with no answer schema:

In [9]:
from shared.settings import lc_chat_model, chat_url

REWRITE = ("Rewrite the last user turn as a standalone search query. "
           "Keep products, account types and conditions. "
           "Do not answer it. Return only the query.")
chat = lc_chat_model("CHAT_MODEL", settings)
query = chat.invoke([("system", REWRITE), ("user", history_text(case["turns"]))]).content
print(query)

Boreal password reset link validity duration.


*Boreal password reset link validity duration.* is not a full sentence, and it does not need to be: it names the
product and the task. Search and answer with it, as in Part A:

In [10]:
hits = lc_pipeline.retrieve(store, query)
show(hits)
answers(query, hits)

  0.7825  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.5801  H02          Atlas reset links expire after 30 minutes. Request a new link if the o
  0.5124  H01          Atlas passwords can be reset by email. Temporary accounts require a su


  3 of 3: Boreal reset links expire after 10 minutes. | cited, quote found: [('H04', True)]


H04 now comes first (about 0.78, with H02 at 0.58), and all three answers say *Boreal reset links expire after 10
minutes.*, citing H04 with its quote found. The rewrite moved the meaning from the earlier turns into the query, and the
search and the answer both got it.

The call is the same for every conversation, so put it in a function, and try it on *What about Boreal?*:

In [11]:
def rewrite(turns):
    return chat.invoke([("system", REWRITE), ("user", history_text(turns))]).content

switch_query = rewrite(switch["turns"])
print(switch_query)
switch_hits = lc_pipeline.retrieve(store, switch_query)
show(switch_hits)
answers(switch_query, switch_hits)

How long does a Boreal reset link remain valid?


  0.6734  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.6418  H02          Atlas reset links expire after 30 minutes. Request a new link if the o
  0.3650  H01          Atlas passwords can be reset by email. Temporary accounts require a su


  3 of 3: A Boreal reset link remains valid for 10 minutes. | cited, quote found: [('H04', True)]


The rewrite is *How long does a Boreal reset link remain valid?*: the task from the Atlas question, the product from
the last turn. H04 leads at about 0.67, and the three answers give Boreal's 10 minutes where Part A refused.

LangChain once shipped this step as a "history-aware retriever", now in the legacy `langchain-classic` package. It
was one chat call before the search, so the labs write the call themselves.

### Checkpoint B

| You should see | Our run |
|---|---|
| Rewrite of *How long is the link valid?* | *Boreal password reset link validity duration.* |
| Its search and answers | H04 first (about 0.78); *Boreal reset links expire after 10 minutes.* 3 of 3, quote found |
| Rewrite of *What about Boreal?* | *How long does a Boreal reset link remain valid?*; H04 first (about 0.67); 10 minutes 3 of 3 |

---

## Part C: Check the rewrite in code

Part C writes two checks that read a query and say whether it kept what the user meant. A rewrite is model output and
can drop or add words on any run; a check in code costs nothing and runs on every query.

- **Keep the product.** The conversation file names the product each follow-up is about; the check passes when each
  one appears in the query.
- **Add no answer.** A query that answers, or copies an earlier answer, brings in numbers the user never typed (*30
  minutes*, a source ID such as *H02*). The check lists every number in the query that is not in a user turn.

Run both on the rewrite from Part B:

In [12]:
import re

def products_kept(query, case):
    return all(product.lower() in query.lower() for product in case["products"])

def new_numbers(query, turns):
    typed = re.findall(r"\d+", " ".join(turn["content"] for turn in turns if turn["role"] == "user"))
    return [number for number in re.findall(r"\d+", query) if number not in typed]

print(f"{query!r}: product kept {products_kept(query, case)} | new numbers {new_numbers(query, case['turns'])}")

'Boreal password reset link validity duration.': product kept True | new numbers []


The rewrite passes both. A check is only useful if it can fail, so try two queries written by hand: the follow-up as
typed, and a rewrite of *What about Boreal?* that carries the Atlas answer into the search:

In [13]:
for bad, bad_case in (("How long is the link valid?", case),
                      ("Do Boreal reset links also expire after 30 minutes?", switch)):
    print(f"{bad!r}: product kept {products_kept(bad, bad_case)} | new numbers {new_numbers(bad, bad_case['turns'])}")

'How long is the link valid?': product kept False | new numbers []
'Do Boreal reset links also expire after 30 minutes?': product kept True | new numbers ['30']


The first fails the product check; the second keeps Boreal and fails the number check with `['30']`, the earlier
Atlas answer. Neither check reads meaning, so a query could name Boreal and still ask the wrong question. They catch
the two failures this lab is about, on every query, without a model call.

### Checkpoint C

| You should see | Our run |
|---|---|
| Part B's rewrite | product kept `True`, new numbers `[]` |
| The two hand-written queries | `False`, `[]`; then `True`, `['30']` |

---

## Part D: Let the model write the query as a tool call (self-study)

Part D gives the model a search tool and lets it write the query itself, in the same call that reads the
conversation. This is the route an agent takes.

A **tool** is a function the model may ask you to run. LangChain's
[`@tool`](https://docs.langchain.com/oss/python/langchain/tools) decorator turns a Python function into one, and the
docstring becomes the description the model reads. Make the search a tool, and print what the model is sent:

In [14]:
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

@tool
def search_documents(query: str) -> str:
    """Search the help-centre documents and return the three closest passages. The query must make sense on its own."""
    return "\n".join(lc_pipeline.evidence(lc_pipeline.retrieve(store, query)))

print(convert_to_openai_tool(search_documents))

{'type': 'function', 'function': {'name': 'search_documents', 'description': 'Search the help-centre documents and return the three closest passages. The query must make sense on its own.', 'parameters': {'properties': {'query': {'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}


The model never sees your code, only this: a name, the description, and a JSON schema with one string argument,
`query`. `bind_tools()` attaches the tool to every call. Send the whole conversation as messages, after a system
message that says to search first:

In [15]:
SYSTEM = ("You answer questions about the Atlas and Boreal products. Search the documents before you answer."
          " Answer only from the passages, citing their IDs.")
with_search = chat.bind_tools([search_documents])
messages = [{"role": "system", "content": SYSTEM}, *case["turns"]]
reply = with_search.invoke(messages)
print("text:", repr(reply.content))
print("tool calls:", reply.tool_calls)

text: ''
tool calls: [{'name': 'search_documents', 'args': {'query': 'link validity'}, 'id': 'call_oaw4sTf10i7cwVOmxkWWOxXi', 'type': 'tool_call'}]


The reply has no text. It holds one **tool call**: the tool's name, the arguments the model wrote and an ID. The
model saw all three turns, Boreal included. Check the query it wrote with Part C's checks:

In [16]:
tool_query = reply.tool_calls[0]["args"]["query"]
print(f"{tool_query!r}: product kept {products_kept(tool_query, case)} | new numbers {new_numbers(tool_query, case['turns'])}")

'link validity': product kept False | new numbers []


*link validity*: the product is gone, and the check says so. Nothing has searched yet: the model asked, and your
code runs the tool. Invoking the tool with the whole tool call returns a `ToolMessage` that carries the call's ID, so
the model can match the result to its request:

In [17]:
tool_message = search_documents.invoke(reply.tool_calls[0])
print(type(tool_message).__name__, "| same ID as the call:", tool_message.tool_call_id == reply.tool_calls[0]["id"])
print(tool_message.content)

ToolMessage | same ID as the call: True
[H02] Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.
[S19-P02-C01] leaky bucket. In case of a shared physical medium (such as radio or 10BASE5), the packets may be delivered according to a multiple access scheme.
[S19-P00-C01] the development of telecommunications in the Bell System. The new concept found little resonance among network implementers until the independent work of Donald Davies at the National Physical Laboratory (United Kingdom) (NPL) in the late 1960s. Davies is credited with coining the modern name packet switching and inspiring numerous packet switching networks in Europe in the decade following, including the incorporation of the concept in the early ARPANET in the United States.


The search found H02 and two chunks about packet switching (*leaky bucket*), and no Boreal policy at all. Send the
conversation back with the model's tool call and the result, and the model writes its answer:

In [18]:
final = with_search.invoke(messages + [reply, tool_message])
print("tool calls:", final.tool_calls)
print(final.content)

tool calls: []
The reset links for Atlas expire after 30 minutes. If the link has expired, you will need to request a new one [H02].


*The reset links for Atlas expire after 30 minutes ... [H02].* The model had the whole conversation in front of
it, and still answered from what its own query found.

A reply can ask for another search instead of answering, so the round trip goes in a loop: run every tool call, send
the results back, stop when a reply has none. `tool_answer()` does that and returns the model's queries with its
final answer. Run it three times:

In [19]:
def tool_answer(turns, model):
    messages = [{"role": "system", "content": SYSTEM}, *turns]
    reply = model.invoke(messages)
    queries = []
    while reply.tool_calls:
        messages.append(reply)
        for call in reply.tool_calls:
            queries.append(call["args"]["query"])
            messages.append(search_documents.invoke(call))
        reply = model.invoke(messages)
    return queries, reply.content

for _ in range(3):
    print(tool_answer(case["turns"], with_search))

(['link validity'], 'The reset links for Atlas expire after 30 minutes. If the link has expired, you will need to request a new one [H02].')


(['link validity'], 'The reset links for Atlas expire after 30 minutes. If the link has expired, you will need to request a new one [H02].')


(['link validity'], 'The reset links for Atlas expire after 30 minutes. If the link has expired, you will need to request a new one [H02].')


The same query, *link validity*, and the same Atlas answer, 3 of 3.

The tool's description is the only instruction the model reads about the query, and *make sense on its own* did not
say what that takes here. Give the same function a description that names the product and the task, and run the loop
again; the search is unchanged:

In [20]:
PRODUCT_RULE = ("Search the help-centre documents and return the three closest passages. Write a query that makes"
                " sense on its own: name the product (Atlas or Boreal) and the task.")
search_named = tool("search_documents", description=PRODUCT_RULE)(search_documents.func)
with_named_search = chat.bind_tools([search_named])
for _ in range(3):
    print(tool_answer(case["turns"], with_named_search))

(['Boreal password reset link validity'], 'The Boreal password reset links expire after 10 minutes [H04].')


(['Boreal password reset link validity'], 'The Boreal password reset links expire after 10 minutes [H04].')


(['Boreal password reset link validity'], 'The Boreal password reset links expire after 10 minutes [H04].')


The query is now *Boreal password reset link validity*, and the answer *The Boreal password reset links expire after
10 minutes [H04].*, 3 of 3.

The two routes put the same step in different places. The rewrite is a call your code makes before every search, so
you can check its query before anything uses it. The tool call lets the model decide when to search and what for,
which helps when a question needs several searches; its query follows the tool's description, not your rewrite
instruction. The bonus lab builds a full agent on this loop.

### Checkpoint D

| You should see | Our run |
|---|---|
| The tool as sent | `search_documents`, the description, one `query` string |
| First reply | no text, one tool call: *link validity*, product kept `False` |
| The loop, first description | *link validity* and Atlas's 30 minutes with H02, 3 of 3 |
| The loop, product rule | *Boreal password reset link validity* and Boreal's 10 minutes with H04, 3 of 3 |

---

## Part E: Test both routes on the seven conversations (self-study)

Part E runs every way of writing the query on every conversation, three times each, so that one run cannot decide.

Four ways: the last turn **as typed**, the **rewrite**, the **tool call** with the first description, and the tool
call with the **product rule**. For the tool routes only the query matters here, so `tool_query()` makes the first
call and reads its query without searching (a reply with no tool call gives an empty query, which fails the product
check). Every query is printed, grouped, with how often it came back:

In [21]:
def tool_query(turns, model):
    calls = model.invoke([{"role": "system", "content": SYSTEM}, *turns]).tool_calls
    return calls[0]["args"]["query"] if calls else ""

routes = {"as typed": lambda turns: turns[-1]["content"], "rewrite": rewrite,
          "tool call": lambda turns: tool_query(turns, with_search),
          "product rule": lambda turns: tool_query(turns, with_named_search)}
queries = {(case["id"], route): Counter(write(case["turns"]) for _ in range(3))
           for case in conversations for route, write in routes.items()}
for (case_id, route), counts in queries.items():
    print(f"{case_id:18}{route:14}" + " | ".join(f"{n}x {q!r}" for q, n in counts.items()))

topic_switch      as typed      3x 'What about Boreal?'
topic_switch      rewrite       1x 'Boreal reset links expire after 30 minutes [H02].' | 2x 'How long does a Boreal reset link remain valid?'
topic_switch      tool call     3x 'Boreal reset link validity'
topic_switch      product rule  3x 'Boreal reset link validity'
link_after_boreal as typed      3x 'How long is the link valid?'
link_after_boreal rewrite       3x 'Boreal password reset link validity duration.'
link_after_boreal tool call     3x 'link validity'
link_after_boreal product rule  3x 'Boreal password reset link validity'
account_type      as typed      3x 'Can temporary accounts do that?'
account_type      rewrite       3x 'Can temporary accounts reset Atlas passwords by email?'
account_type      tool call     3x 'temporary accounts reset password email'
account_type      product rule  3x 'temporary accounts reset Atlas password by email'
pronoun           as typed      3x 'Who can export all of them?'
pronoun      

Now count, per conversation and route, how many of the three queries keep the product, add no new number, and put
the answering policy first. The last count needs a policy, so it is blank for the two conversations with none:

In [22]:
def first_id(query):
    return lc_pipeline.retrieve(store, query, k=1)[0][0].id if query else None

print(f"{'conversation':18}{'route':14}{'product kept':>13}{'no new number':>15}{'right policy first':>20}")
for (case_id, route), counts in queries.items():
    case = by_id[case_id]
    kept = sum(n for q, n in counts.items() if products_kept(q, case))
    clean = sum(n for q, n in counts.items() if not new_numbers(q, case["turns"]))
    right = sum(n for q, n in counts.items() if first_id(q) == case["source_id"])
    print(f"{case_id:18}{route:14}{kept:>8} of 3{clean:>10} of 3{(f'{right} of 3' if case['source_id'] else '-'):>20}")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

conversation      route          product kept  no new number  right policy first


topic_switch      as typed             3 of 3         3 of 3              3 of 3


topic_switch      rewrite              3 of 3         2 of 3              2 of 3
topic_switch      tool call            3 of 3         3 of 3              3 of 3


topic_switch      product rule         3 of 3         3 of 3              3 of 3


link_after_boreal as typed             0 of 3         3 of 3              0 of 3


...

evidence_gap      product rule         3 of 3         3 of 3                   -


two_products      as typed             0 of 3         3 of 3                   -


two_products      rewrite              3 of 3         3 of 3                   -


two_products      tool call            0 of 3         3 of 3                   -


two_products      product rule         3 of 3         3 of 3                   -

gpt-4o-mini at https://api.openai.com/v1


Route by route:

- **As typed** fails the product check on five conversations, yet puts the right policy first on three of them: only
  Atlas has temporary accounts, report exports and invoices here, so the topic was enough. With a third product it
  would not be. `link_after_boreal` gets the wrong policy, 0 of 3.
- **The rewrite** keeps the product in all 21 queries and puts the right policy first in 14 of the 15 that have one.
  The miss is a rewrite that answered instead of rewriting: for `topic_switch` one run wrote *Boreal reset links
  expire after 30 minutes [H02].*, the earlier Atlas answer turned into a Boreal claim. The number check caught it.
- **The tool call** drops the product on `link_after_boreal`, `account_type` and `two_products`, 3 of 3 each.
- **The product rule** passes every column.

The queries also show an earlier answer leaking in. For `earlier_answer`, all 3 rewrites are *Atlas invoice download
location for monthly invoices [H05].*: the assistant's *monthly* and its citation, copied into the search. The number
check catches the *05*; it would not catch *monthly* alone, since a check sees only what it tests.

Does an earlier answer reach a reply? In `evidence_gap` the user learned that Atlas invoices are sent *monthly*, then
asks *What about Boreal?* No policy covers Boreal's billing, so the right answer is a refusal; *monthly* for Boreal
would be the earlier answer reused. Run both routes:

In [23]:
gap = by_id["evidence_gap"]
gap_query = rewrite(gap["turns"])
print(gap_query)
gap_hits = lc_pipeline.retrieve(store, gap_query)
show(gap_hits)
answers(gap_query, gap_hits)
for _ in range(3):
    print(tool_answer(gap["turns"], with_named_search))

How often are Boreal invoices sent?


  0.5380  H05          Atlas invoices are sent monthly. Workspace owners can download invoice
  0.3801  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.2491  H03          Atlas exports reports as CSV files. Only workspace owners can export a


  3 of 3: Insufficient evidence. | cited, quote found: []


(['Boreal invoices frequency'], 'The documents do not specify how often Boreal invoices are sent. Only the frequency for Atlas invoices is mentioned, which are sent monthly [H05].')


(['Boreal invoices frequency'], 'The documents do not specify how often Boreal invoices are sent. The only information available is about Atlas invoices, which are sent monthly [H05].')


(['Boreal invoices frequency'], 'The documents do not specify how often Boreal invoices are sent. Only the frequency for Atlas invoices is mentioned, which are sent monthly [H05].')


The rewrite asks *How often are Boreal invoices sent?*, the search puts Atlas's billing policy H05 first (about 0.54),
and the answer refuses 3 of 3. The tool route says 3 of 3 that the documents do not give Boreal's billing, and adds
that Atlas invoices are sent monthly, citing H05. *Monthly* reached the reply as Atlas's fact, not as Boreal's.

### Checkpoint E

| You should see | Our run |
|---|---|
| As typed | product kept 0 of 3 on five conversations; wrong policy first on `link_after_boreal` only |
| Rewrite | product kept 21 of 21; *[H05]* in 3 of 3 `earlier_answer` queries; one `topic_switch` rewrite answered instead |
| Tool call | product dropped on three conversations; wrong policy first on `link_after_boreal` |
| Product rule | 3 of 3 in every column |
| `evidence_gap` | rewrite: *Insufficient evidence.* 3 of 3; tool: no Boreal billing, Atlas's monthly cited to H05 |

---

## Part F: Ask which product instead of guessing (self-study)

Part F handles the follow-up no query can resolve: the conversation does not say which product the user means.

In `two_products` the user asked about their Atlas password, then about a colleague's Boreal account, then *How long
is the link valid?* Which link? In Part E the query as typed and the first tool description named no product, so the
search puts Atlas first, as in Parts A and D; the rewrite and the product rule named both. Each route chose a reading
without saying so. The honest reply is a question: *Which product do you mean, Atlas or Boreal?*

The first idea is to tell the rewrite step to ask. Add one line to the instruction and run it on all seven
conversations, where only `two_products` should ask:

In [24]:
ASK = (" If the conversation does not make clear which product the last turn is about,"
       " return only: Which product do you mean, Atlas or Boreal?")
for case in conversations:
    print(f"{case['id']:18}{chat.invoke([('system', REWRITE + ASK), ('user', history_text(case['turns']))]).content}")

topic_switch      Which product do you mean, Atlas or Boreal?


link_after_boreal How long is the password reset link valid for Boreal accounts?


account_type      Can temporary accounts reset their Atlas password by email?


pronoun           Which product do you mean, Atlas or Boreal?


earlier_answer    Which product do you mean, Atlas or Boreal?


evidence_gap      Which product do you mean, Atlas or Boreal?


two_products      Which product do you mean, Atlas or Boreal?


It asked on five of the seven, `two_products` and four that were clear, among them *What about Boreal?*, which names
its product. A rule the model applies with judgement fired where no judgement was needed.

This rule needs none. Ask when the user has named more than one product and the last turn names none. Code can test
that exactly, on every turn:

In [25]:
PRODUCTS = ("Atlas", "Boreal")

def named(text):
    return {product for product in PRODUCTS if product.lower() in text.lower()}

def needs_question(turns):
    said = [turn["content"] for turn in turns if turn["role"] == "user"]
    return len(named(" ".join(said))) > 1 and not named(said[-1])

for case in conversations:
    print(f"{case['id']:18}{'ask which product' if needs_question(case['turns']) else 'rewrite and search'}")

topic_switch      rewrite and search
link_after_boreal rewrite and search
account_type      rewrite and search
pronoun           rewrite and search
earlier_answer    rewrite and search
evidence_gap      rewrite and search
two_products      ask which product


Only `two_products` asks. The rule reads the user's turns, because the user's words say what they mean. A real help
desk would also know the customer's product from their account, which beats any rule (Lab 05's metadata filter).

Put the steps together: ask when the rule says so, otherwise rewrite, search and answer. Run it on Part A's
conversation and on `two_products`:

In [26]:
def answer_follow_up(turns):
    if needs_question(turns):
        return "Which product do you mean, Atlas or Boreal?"
    query = rewrite(turns)
    return lc_pipeline.answer(query, lc_pipeline.evidence(lc_pipeline.retrieve(store, query)), settings).answer

for case_id in ("link_after_boreal", "two_products"):
    print(f"{case_id:18}{answer_follow_up(by_id[case_id]['turns'])}")

link_after_boreal Boreal reset links expire after 10 minutes.
two_products      Which product do you mean, Atlas or Boreal?


*Boreal reset links expire after 10 minutes.* for the conversation that opened the lab, and a question back for the
one that cannot be answered without one.

### Checkpoint F

| You should see | Our run |
|---|---|
| The ask line in the instruction | asks on 5 of 7: `topic_switch`, `pronoun`, `earlier_answer`, `evidence_gap`, `two_products` |
| `needs_question()` | `ask which product` for `two_products` only |
| `answer_follow_up()` | Boreal's 10 minutes for `link_after_boreal`; *Which product do you mean, Atlas or Boreal?* for `two_products` |

---

## What you can do now

- Explain why a follow-up searched as typed can retrieve the wrong product, or leave the answer with no question,
  while every citation check passes.
- Rewrite a follow-up into a standalone query with one chat call, and check the query in code.
- Give the model a search tool with `@tool` and `bind_tools`, run its calls in a loop, and read the query it wrote.
- Test both routes on a set of conversations over several runs, and find where an earlier answer leaks in.
- Decide in code when to ask which product the user means, and say why a prompt line did not.

## Challenge

Use only what this lab showed. A worked solution is in `solved/` for the instructor. Task 1 is the challenge; task 2 is
optional. For each task, keep the printed results and your answer to **Explain**.

1. **Add a conversation and test it on every route.** A user asks *Can I reset my Atlas password by email?*, gets
   *Yes. Atlas passwords can be reset by email [H01].*, and asks *Does Boreal work that way too?* H04 says Boreal's
   reset uses a mobile app, so the right answer is no.

   - Under **Task 1**, write the conversation as a dictionary shaped like those in `conversations.json` (`products`
     `["Boreal"]`, `source_id` `"H04"`). The cell answers it as typed, through the rewrite and through the tool with
     the product rule.

   **Check:** each route's answers. Our worked solution got *Yes, Boreal works that way too.* as typed (3 of 3, citing
   H04, quotes found); *Insufficient evidence.* through the rewrite (3 of 3), whose query passed both checks; and *No,
   Boreal does not work the same way. Password resets for Boreal use a mobile app ...* through the tool (3 of 3).

   **Explain:** which route answered correctly, and why did the other two not? Would any check in this lab have
   flagged the answer as typed?

2. **Optional: check a rewrite instruction from an older tutorial.** Tutorials for LangChain's history-aware
   retriever often used: *Given a chat history and the latest user question which might reference context in the chat
   history, formulate a standalone question which can be understood without the chat history. Do NOT answer the
   question, just reformulate it if needed and otherwise return it as is.*

   - Under **Task 2**, set `OLD_REWRITE` to it. The cell rewrites each conversation three times and checks each query.

   **Check:** the queries that fail. Our worked solution got 6 of 21, all on the product: *How long is the password
   reset link valid for?* for `link_after_boreal` and `two_products`, 3 of 3 each.

   **Explain:** which part of this lab's `REWRITE` does the older instruction lack, and which conversation shows what
   it costs?

**Task 1.** Your conversation, then every route:

In [27]:
new_case = None    # TODO: {"id": ..., "slice": ..., "turns": [...], "products": ["Boreal"], "source_id": "H04"}

if new_case is None:
    print("⚠️ Write the conversation, then run the cell again.")
else:
    last = new_case["turns"][-1]["content"]
    print("as typed:"); answers(last, lc_pipeline.retrieve(store, last))
    new_query = rewrite(new_case["turns"])
    print(f"rewrite: {new_query!r} | product kept {products_kept(new_query, new_case)}"
          f" | new numbers {new_numbers(new_query, new_case['turns'])}")
    answers(new_query, lc_pipeline.retrieve(store, new_query))
    print("tool, product rule:")
    for _ in range(3):
        print(" ", tool_answer(new_case["turns"], with_named_search))

⚠️ Write the conversation, then run the cell again.


**Task 2 (optional).** The older instruction, then the checks:

In [28]:
OLD_REWRITE = None    # TODO: the instruction from the task, as a string

if OLD_REWRITE is None:
    print("⚠️ Set OLD_REWRITE, then run the cell again.")
else:
    for case in conversations:
        for _ in range(3):
            old = chat.invoke([("system", OLD_REWRITE), ("user", history_text(case["turns"]))]).content
            print(f"{case['id']:18}product kept {products_kept(old, case)!s:6}"
                  f"new numbers {new_numbers(old, case['turns'])!s:10}{old!r}")

⚠️ Set OLD_REWRITE, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ValueError: Function must have a docstring if description not provided.` | `@tool` on a function with no docstring | Add a docstring, or pass `description=` as Part D does for `search_named`; the model reads it |
| `IndexError: list index out of range` at `reply.tool_calls[0]` | the model answered in text and called no tool (it does this for a turn such as *Thanks, that is all.*) | Read `reply.tool_calls` before you index it, as `tool_query()` and `tool_answer()` do |
| `OpenAIInvalidRequestError: Error code: 400 ... An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'.` | the model's tool call was sent back without a `ToolMessage` for each call | Append a result for every call before the next `invoke()`, as `tool_answer()` does |
| `OpenAIInvalidRequestError: Error code: 400 ... Function tools with reasoning_effort are not supported for gpt-6-luna in /v1/chat/completions.` | `REASONING_EFFORT` is `low`, `medium` or `high` for a reasoning model | Set `REASONING_EFFORT=none` in `.env`: Chat Completions takes tools on `gpt-6-luna` only at that effort |

## Resources

- [LangChain: retrieval](https://docs.langchain.com/oss/python/langchain/retrieval): 2-step RAG, agentic RAG with
  retrieval as a tool, and query rewriting as a step between them
- [LangChain: tools](https://docs.langchain.com/oss/python/langchain/tools): `@tool`, descriptions and argument
  schemas
- [LangChain: models, tool calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling): `bind_tools`,
  `tool_calls`, the execution loop and `ToolMessage`
- [OpenAI: function calling](https://developers.openai.com/api/docs/guides/function-calling): what the API receives
  for a tool and returns for a call

Next: [Lab 10](../10_Long_Context_Or_Retrieval/README.md), *How to choose between long context and retrieval*. Back to [the lab list](../README.md).